# Lab 5: From today's lab to a real agent

You already ran a **tool-using agent** on governed data, measured it, and fixed instructions. This short wrap explains what lived under the hood and what you would do next outside the classroom.

This notebook does **not** deploy an App or serving endpoint. It is the mental model for production.

### Docs
- [AI Playground](https://learn.microsoft.com/en-us/azure/databricks/large-language-models/ai-playground)
- [Migrate to Unity Gateway](https://docs.databricks.com/aws/en/ai-gateway/migrate-to-unity-gateway)
- [Author agents on Databricks Apps](https://docs.databricks.com/aws/en/generative-ai/agent-framework/author-agent)
- [Deploy agents (Model Serving path)](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/deploy-agent)


## Load config

Confirms your catalog, schema, and model widget (UC FQN or legacy endpoint).


In [ ]:
%run ./_config


## What you leave with today

Honest frame:

| You have | You do **not** have yet |
|---|---|
| A working **pattern**: model + instructions + UC tools + eval | A shipped product for all analysts |
| Governed synthetic tables and three read-only tools | An App chat UI or production endpoint |
| Traces and an eval set you can re-run | CI, monitoring, and least-privilege service identities |

That is enough to prove the idea. Packaging comes next.


## What is in `_agent_core` (plain language)

Labs 2 and 3 load `_agent_core`. Think of it as a small **agent kit**, not a product:

| Piece | Plain meaning |
|---|---|
| Instructions (`INSTRUCTIONS_V1`) | Written rules for the model |
| Tool list (`TOOLS`) | What the model is allowed to ask for |
| `call_tool` | Runs your Unity Catalog functions **as you** (filters and masks apply) |
| `run_agent` | The loop: ask model → run tools → answer in English |
| LLM client | Calls your model via **Unity Gateway** (`system.ai…`) or a **legacy serving endpoint** |

You customize the **rules** and **tools** for your use case. You do not need to invent the loop from scratch in a two-hour room.


## Where you already called the agent


In [ ]:
print(f"Catalog.schema:  {FQ_PLAIN}")
print(f"Model widget:    {LLM_ENDPOINT}")
print(f"Access mode:     {LLM_ACCESS_MODE}  (unity_gateway or legacy_endpoint)")
print()
print("Call sites in this workshop:")
print("  Lab 2  → run_agent(...) on demo questions; Playground uses the same tools + instructions")
print("  Lab 3  → MLflow evaluate calls run_agent with INSTRUCTIONS_V1 then V2")
print("  _agent_core → defines run_agent (the agent itself)")


## Map lab → production (what changes next)

Same ideas, different packaging:

| Today (this workshop) | Next (outside the room) |
|---|---|
| Lab 1 UC functions | Same tools (or MCP); grant `EXECUTE` to the app / endpoint identity |
| `_agent_core` loop + instructions | Package as a real agent (for example MLflow `ResponsesAgent` / Agent Bricks or Apps template) |
| Lab 2 Playground / `run_agent` | Chat UI on Databricks Apps, or an API others can call |
| Lab 3 MLflow eval + V2 instructions | Eval in CI + production monitoring / review |
| You = identity | App service principal and/or on-behalf-of-user auth |
| Model widget (Gateway or legacy) | Prefer UC model services under Unity Gateway; same grant story as data |

### Four layers

1. **Prove the pattern** — notebook + Playground (done)
2. **Register / serve** — package the agent so others can call it
3. **App or existing UI** — analysts use a chat box, not a notebook
4. **Ops** — permissions, tracing, eval gates, monitoring

Docs for later (not run in class): authoring on Apps, Unity Gateway, and (older path) Model Serving `agents.deploy()`.


## Optional: peek at the loaded agent kit

Install lab packages (same as Labs 2–3), load `_agent_core`, then show tool names. No deploy.


### Install packages for the agent kit

Needed so `%run ./_agent_core` can import MLflow tracing types on serverless.


In [ ]:
%pip install -U -q "mlflow[databricks]>=3.4" "databricks-sdk>=0.60" openai httpx
dbutils.library.restartPython()


### Reload config after restart

`restartPython` clears notebook state; load widgets again before the agent kit.


In [ ]:
%run ./_config


### Load the agent kit

Defines instructions, tools, and `run_agent`.


In [ ]:
%run ./_agent_core


### Tools and first instruction line

Confirms the agent kit loaded from `_agent_core`.


In [ ]:
print("Tools:", [t["function"]["name"] for t in TOOLS])
print("Instructions (first line):", INSTRUCTIONS_V1.strip().splitlines()[0])


## Close

You measured quality, fixed instructions, and saw governance apply through tools.  

If you have not yet filled the ideation canvas, open Lab 4 or the README image and sketch one use case: problem, data, pattern, value, three test questions.

**Takeaway:** today's agent is real as a demo of the pattern. A production agent reuses that pattern inside a packaged, governed service — that is the path above, not a different idea.
